# ETL Transform: Stocks

This notebook runs the **stocks ETL pipeline**: ingest from Postgres (with warmup window) → transform (returns, volatility, technical indicators) → save to `historical_processed` → publish to S3.

**S3 upload modes (set in config cell below):**
- **Per day**: one CSV per book per day at `stocks/transformed/crypto/book={book}/year=.../month=.../day=.../format=csv/YYYYMMDD-{book}.csv`
- **Batch (run/week/month/year)**: one CSV per run (all books) or per (book, partition) at `stocks/transformed/crypto/book={book}/year=.../week=...` or `month=...` or `year=.../format=csv/...`

Set `AWS_STOCKS_BUCKET` or `AWS_DEFAULT_BUCKET` in `.env` for S3 uploads.

In [1]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path(".").resolve()
project_root = _cwd if (_cwd / "src").is_dir() else (_cwd.parent.parent if _cwd.name == "etl" else _cwd)
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

import pandas as pd

In [2]:
# Config: date range, books, and S3 upload options
SINCE = "2025-01-01"
UNTIL = "2025-12-31"
BOOKS = ["btc-usd"]  # None = all books; or e.g. ["btc-usd", "eth-usd"]
WARMUP_DAYS = 252
# S3: per-day (one file per book per day) and/or batch (one file per book per week/month/year)
UPLOAD_S3 = True
UPLOAD_S3_BATCH = ["year"]  # e.g. ["run", "week", "month", "year"] or None

In [3]:
# Run stocks ETL: transform (with warmup for indicators), save to Postgres, publish to S3.

import logging
# Show pipeline progress in the notebook (ingest, transform, save, S3 upload)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S", force=True)
for _name in ("pipelines.etl_transform", "pipelines.etl_cli", "transform.stocks.stock_transformers"):
    logging.getLogger(_name).setLevel(logging.INFO)

from pipelines.etl_transform import run_stocks_etl

transformed_df = run_stocks_etl(
    since=SINCE,
    until=UNTIL,
    books=BOOKS,
    warmup_days=WARMUP_DAYS,
    stocks_bucket=None,  # uses AWS_STOCKS_BUCKET or AWS_DEFAULT_BUCKET from .env
    save_to_postgres=False,
    upload_s3=UPLOAD_S3,
    upload_s3_batch=UPLOAD_S3_BATCH,
)

print(f"Transformed {len(transformed_df)} stock records")

20:39:20 - INFO - Ingesting stocks (warmup 2024-04-24 to 2025-12-31)...
20:39:20 - INFO - ============================================================
20:39:20 - INFO - INGEST STOCKS
20:39:20 - INFO - ============================================================
20:39:20 - INFO - Fetching stocks from database...
20:39:20 - INFO -   Books: ['btc-usd']
20:39:20 - INFO - Retrieved 4115 records
20:39:20 - INFO - Filtered since 2024-04-24: 608 records
20:39:20 - INFO - Filtered until 2025-12-31: 534 records
20:39:20 - INFO - Ingestion complete: 534 records
20:39:20 - INFO - Transforming (returns, volatility, technical indicators)...
20:39:20 - INFO - Stock transformation pipeline initialized
20:39:20 - INFO - Transforming 534 stock records...


Connection to the database successful!
Table name set to: historical
Connection closed.


20:39:21 - INFO - Stock transformation complete: 534 records
20:39:21 - INFO - Transformed 364 stock records (2025-01-01 to 2025-12-31)
20:39:21 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
20:39:21 - INFO - Uploading 364 book/day files to s3://test-financial-stocks-bucket/...
20:41:03 - INFO - Uploaded 364 group files to s3://test-financial-stocks-bucket/
20:41:04 - INFO - Uploaded stocks batch year (book=btc-usd, partition 2025-01-01) to s3://test-financial-stocks-bucket/stocks/transformed/crypto/book=btc-usd/year=2025/format=csv/y2025-btc-usd.csv


Transformed 364 stock records


In [4]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

,ref,book,date,open,high,low,close,adj_close,volume,simple_return,...,sma_200,ema_12,ema_26,rsi_14,macd,macd_signal,macd_histogram,bb_upper,bb_middle,bb_lower
170,https://finance.yahoo.com,btc-usd,2025-01-01,93425.10,94929.87,92788.13,94419.76,94419.76,24519888919,0.010602,...,NaN,95438.365498,96201.112866,36.569337,-762.747368,72.446092,-835.193460,106274.908118,97936.3790,89597.849882
171,https://finance.yahoo.com,btc-usd,2025-01-02,94416.29,97739.82,94201.57,96886.88,96886.88,46009564411,0.026129,...,NaN,95661.213883,96251.910431,48.551073,-590.696549,-60.182436,-530.514112,105888.842119,97707.7600,89526.677881
172,https://finance.yahoo.com,btc-usd,2025-01-03,96881.73,98956.91,96034.62,98107.43,98107.43,35611391163,0.012598,...,NaN,96037.554824,96389.356325,50.806156,-351.801502,-118.506249,-233.295252,105545.940515,97544.4830,89543.025485
173,https://finance.yahoo.com,btc-usd,2025-01-04,98106.99,98734.43,97562.98,98236.23,98236.23,22342608078,0.001313,...,NaN,96375.812543,96526.161783,52.363472,-150.349240,-124.874847,-25.474392,104598.878686,97241.3595,89883.840314
174,https://finance.yahoo.com,btc-usd,2025-01-05,98233.91,98813.30,97291.77,98314.96,98314.96,20525254825,0.000801,...,NaN,96674.142921,96658.665354,58.291394,15.477567,-96.804365,112.281931,102978.462814,96855.6215,90732.780186


['ref', 'book', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'simple_return', 'log_return', 'volatility_20d', 'volatility_60d', 'volatility_parkinson', 'volatility_gk', 'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower']


## Optional: Specific books and date range

In [5]:
# transformed_df = run_stocks_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     books=["btc-usd", "eth-usd"],
#     warmup_days=252,
#     save_to_postgres=True,
#     upload_s3=True,
# )